# SCRFD Competition Offline Runner

Notebook template for Kaggle competition environments with internet disabled.

Expected Kaggle Dataset inputs:
- `scrfd-fullsearch-kaggle-src`: contains `scrfd-fullsearch-kaggle-offline.zip`
- `scrfd-wheelhouse`: optional wheel bundle if the competition image is missing packages
- `widerface-retinaface-format`: your WIDER FACE dataset in SCRFD format

In [ ]:
REPO_ZIP = "/kaggle/input/scrfd-fullsearch-kaggle-src/scrfd-fullsearch-kaggle-offline.zip"
DATA_ROOT = "/kaggle/input/widerface-retinaface-format/retinaface"
WHEELHOUSE = "/kaggle/input/scrfd-wheelhouse"

RUN_MODE = "baseline_train_quick"
# Options: baseline_train_quick, baseline_train, baseline_eval, step1_generate, step1_train, step1_eval,
# step1_stat, step2_generate, step2_train, step2_eval, step2_stat, full_search

GPU_ID = 0
TOTAL_EPOCHS = 16
EVAL_INTERVAL = 4
CHECKPOINT_INTERVAL = 4
IDX_FROM = 1
IDX_TO = 64
TEMPLATE = 10
CHECKPOINT = ""
THR = "0.02"
MODE_VALUE = "0"


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

repo_root = Path('/kaggle/working/scrfd-fullsearch-kaggle')
if repo_root.exists():
    shutil.rmtree(repo_root)

repo_source = Path(REPO_ZIP)
if repo_source.is_dir():
    source_root = repo_source / 'scrfd-fullsearch-kaggle'
    if source_root.exists():
        shutil.copytree(source_root, repo_root)
    else:
        shutil.copytree(repo_source, repo_root)
else:
    with zipfile.ZipFile(repo_source) as zf:
        zf.extractall('/kaggle/working')

assert repo_root.exists(), f'Missing repo after extraction: {repo_root}'
os.chdir(repo_root)
print('Repo ready at', repo_root)


In [ ]:
import subprocess
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/scrfd-fullsearch-kaggle')
entry = repo_root / 'scripts' / 'kaggle_competition_entry.py'
work_root = repo_root / 'work_dirs'
result_root = repo_root / 'wouts'

cmd = [
    sys.executable,
    str(entry),
    '--mode', RUN_MODE,
    '--data-root', DATA_ROOT,
    '--work-root', str(work_root),
    '--result-root', str(result_root),
    '--gpu-id', str(GPU_ID),
    '--idx-from', str(IDX_FROM),
    '--idx-to', str(IDX_TO),
    '--template', str(TEMPLATE),
    '--thr', str(THR),
    '--mode-value', str(MODE_VALUE),
]

if TOTAL_EPOCHS:
    cmd.extend(['--total-epochs', str(TOTAL_EPOCHS)])
if EVAL_INTERVAL:
    cmd.extend(['--eval-interval', str(EVAL_INTERVAL)])
if CHECKPOINT_INTERVAL:
    cmd.extend(['--checkpoint-interval', str(CHECKPOINT_INTERVAL)])

if CHECKPOINT:
    cmd.extend(['--checkpoint', CHECKPOINT])

if Path(WHEELHOUSE).exists():
    cmd.extend(['--wheelhouse', WHEELHOUSE])
else:
    cmd.append('--skip-offline-install')

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)
